# 4.4 Feature iteration and split check

Owner: Jay Malhotra

After the benchmark (4.1) and the error analysis (4.3), two questions were open:

1. Jingyu found that most missed late orders are short, within-state orders, and that lateness comes in waves. Can extra features fix that?
2. How much would a random split have flattered our results compared with the time split we use?

Rules, same as the rest of the project:
- Training period only (orders bought before 1 April 2018). The test set is not loaded.
- Same pipeline as section 3.4 and same models as section 4.1.
- Same 5 time-series folds with the 2,000-order gap (`src/evaluation.py`).
- Every new feature uses only information available before the order was bought (D10).

Models are compared on **lift**: PR-AUC divided by the late rate of the fold, so 1.0 means random. The late rate swings from 2.8% to 18.8% between folds, so raw PR-AUC would mostly reflect which fold had more late orders.

## 0. Setup

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import MissingIndicator, SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

# Make src importable whether the notebook runs from wip/ or the repo root.
ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src import config
from src.evaluation import cv_splitter, recall_at_budget

RS = config.RANDOM_STATE
RAW, PROC = ROOT / "data" / "raw", ROOT / "data" / "processed"

## 1. Data

`olist_final.csv` is the modelling table (one row per order, approval-time features only). `olist_merged_modeling.csv` holds every delivered order with its outcome, which is the pool the history features look back into. The raw item and seller tables give each order its sellers and the seller states.

In [2]:
df = pd.read_csv(PROC / "olist_final.csv",
                 parse_dates=["order_purchase_timestamp", "order_estimated_delivery_date"])
pool = pd.read_csv(PROC / "olist_merged_modeling.csv",
                   parse_dates=["order_purchase_timestamp", "order_delivered_customer_date",
                                "order_estimated_delivery_date"])
items = pd.read_csv(RAW / "olist_order_items_dataset.csv.gz", usecols=["order_id", "seller_id"]).drop_duplicates()
sellers = pd.read_csv(RAW / "olist_sellers_dataset.csv.gz", usecols=["seller_id", "seller_state"])

# Training period only. Sorted by purchase time because TimeSeriesSplit assumes time order.
train = (df[df["order_purchase_timestamp"] < config.TEST_START]
         .sort_values("order_purchase_timestamp", kind="stable").reset_index(drop=True))
assert len(train) == 64_038, "population must match DECISIONS.md"
print(f"Training orders: {len(train):,}, late rate {train[config.TARGET].mean():.1%}")
print(f"History pool: {len(pool):,} delivered orders")

Training orders: 64,038, late rate 8.0%
History pool: 96,456 delivered orders


## 2. New features, built only from the past

Each feature is computed from orders whose outcome was already known when the current order was bought. That is what makes them legal under D10.

- **`sys_late_14d`**: share of all orders promised in the last 14 days that missed their promise. A promise for day *e* is known to be kept or missed by the end of day *e*, so only promises up to the day before purchase count.
- **`state_late_30d`**: the same thing within the customer's state, over 30 days.
- **`seller_late_rate`, `seller_past_orders`**: the seller's late rate on orders **delivered** before this purchase, pulled towards 7% for sellers with little history so a new seller isn't scored 0% or 100%.
- **`route_slack_days`, `route_slack_ratio`, `route_hist_days`**: promised days compared with how long the same seller-state to customer-state route has taken on orders delivered before this purchase.
- **`promised_days_per_100km`**: promised days per 100 km of distance (Mia's suggestion). It uses no history, so it is legal by construction.

In [3]:
DAY = np.timedelta64(1, "D")


def recent_late_rate(query_times, est_dates, late, window_days):
    """Share of orders promised in [t - window, t - 1 day] that missed the promise."""
    order = np.argsort(est_dates.values)
    e = est_dates.values[order]
    cum = np.concatenate([[0], np.cumsum(late.values[order])])
    t = query_times.values
    hi = np.searchsorted(e, t - DAY, side="right")
    lo = np.searchsorted(e, t - window_days * DAY, side="left")
    n = hi - lo
    return np.where(n > 0, (cum[hi] - cum[lo]) / np.maximum(n, 1), np.nan)


def past_group_stats(query, events, key, value, event_time):
    """Running count and sum of `value` per group, using only events strictly before purchase."""
    ev = events.sort_values(event_time).copy()
    ev["n"] = ev.groupby(key).cumcount() + 1
    ev["s"] = ev.groupby(key)[value].cumsum()
    q = query.sort_values("order_purchase_timestamp")
    out = pd.merge_asof(q, ev[[key, event_time, "n", "s"]], left_on="order_purchase_timestamp",
                        right_on=event_time, by=key, direction="backward", allow_exact_matches=False)
    out[["n", "s"]] = out[["n", "s"]].fillna(0)
    return out

Before using them on real data, the functions are checked on a tiny made-up example where the right answer is known. If either check fails, the notebook stops.

In [4]:
T = pd.to_datetime
# Promise for 10 Jan must NOT count for an order bought on 10 Jan, but must count on 11 Jan.
r = recent_late_rate(pd.Series(T(["2018-01-02", "2018-01-10", "2018-01-11"])),
                     pd.Series(T(["2018-01-01", "2018-01-05", "2018-01-10"])), pd.Series([1, 0, 1]), 14)
assert np.allclose(r, [1.0, 0.5, 2 / 3]), r

# An order bought on 5 Jan may only see the delivery from 3 Jan, not the one from 20 Jan.
q = pd.DataFrame({"order_id": ["a", "b"], "seller_id": ["s", "s"],
                  "order_purchase_timestamp": T(["2018-01-05", "2018-02-01"])})
ev = pd.DataFrame({"seller_id": ["s", "s"], "delivered": T(["2018-01-20", "2018-01-03"]), "late": [1, 0]})
chk = past_group_stats(q, ev, "seller_id", "late", "delivered").set_index("order_id")
assert chk.loc["a", "n"] == 1 and chk.loc["b", "n"] == 2, chk
print("Point-in-time checks passed")

Point-in-time checks passed


In [5]:
t = train["order_purchase_timestamp"]
pool_late = pool[config.TARGET]

# System-wide and state-level recent lateness
train["sys_late_14d"] = recent_late_rate(t, pool["order_estimated_delivery_date"], pool_late, 14)
train["state_late_30d"] = np.nan
for state, idx in train.groupby("customer_state").groups.items():
    p = pool[pool["customer_state"] == state]
    if len(p):
        train.loc[idx, "state_late_30d"] = recent_late_rate(
            train.loc[idx, "order_purchase_timestamp"], p["order_estimated_delivery_date"], p[config.TARGET], 30)

# Seller history: one row per (order, seller); multi-seller orders take the riskiest seller
events = items.merge(pool[["order_id", "order_delivered_customer_date", config.TARGET]], on="order_id")
query = items.merge(train[["order_id", "order_purchase_timestamp"]], on="order_id")
sh = past_group_stats(query, events, "seller_id", config.TARGET, "order_delivered_customer_date")
sh["rate"] = (sh["s"] + 5 * 0.07) / (sh["n"] + 5)   # 5 pseudo-orders at the 7% overall rate
sh = sh.groupby("order_id").agg(seller_late_rate=("rate", "max"), seller_past_orders=("n", "min"))
train = train.join(sh, on="order_id")

# Route slack: promised days vs recent delivery time on the same seller-state -> customer-state route
first_seller = items.drop_duplicates("order_id").merge(sellers, on="seller_id")[["order_id", "seller_state"]]
pool_r = pool.merge(first_seller, on="order_id", how="left")
pool_r["route"] = pool_r["seller_state"].fillna("?") + ">" + pool_r["customer_state"]
pool_r["days"] = (pool_r["order_delivered_customer_date"] - pool_r["order_purchase_timestamp"]).dt.total_seconds() / 86400
q = train[["order_id", "order_purchase_timestamp", "customer_state"]].merge(first_seller, on="order_id", how="left")
q["route"] = q["seller_state"].fillna("?") + ">" + q["customer_state"]
rs = past_group_stats(q[["order_id", "route", "order_purchase_timestamp"]], pool_r, "route", "days",
                      "order_delivered_customer_date")
rs["route_hist_days"] = (rs["s"] + 10 * 12.5) / (rs["n"] + 10)  # thin routes pulled towards ~12.5 days
train = train.merge(rs[["order_id", "route_hist_days"]], on="order_id", how="left")
promised_from_purchase = (train["order_estimated_delivery_date"] - t.dt.normalize()).dt.days
train["route_slack_days"] = promised_from_purchase - train["route_hist_days"]
train["route_slack_ratio"] = promised_from_purchase / train["route_hist_days"]

# Promised days per 100 km (distance floored at 1 km so same-zip orders don't divide by zero)
train["promised_days_per_100km"] = train["promised_days_from_approval"] / (train["mean_distance_km"].clip(lower=1) / 100)

train = train.sort_values("order_purchase_timestamp", kind="stable").reset_index(drop=True)
assert len(train) == 64_038
NEW = ["sys_late_14d", "state_late_30d", "seller_late_rate", "seller_past_orders",
       "route_slack_days", "route_slack_ratio", "route_hist_days", "promised_days_per_100km"]
train[NEW].describe().T.round(3)

,count,mean,std,min,25%,50%,75%,max
sys_late_14d,63989.0,0.059,0.042,0.000,0.032,0.045,0.070,0.245
state_late_30d,62678.0,0.058,0.065,0.000,0.026,0.036,0.066,1.000
seller_late_rate,64038.0,0.042,0.032,0.002,0.020,0.035,0.057,0.535
seller_past_orders,64038.0,133.200,216.218,0.000,10.000,44.000,152.000,1421.000
route_slack_days,64038.0,13.086,6.499,-4.595,9.238,12.264,16.164,140.016
route_slack_ratio,64038.0,2.128,0.578,0.723,1.763,2.055,2.395,13.135
route_hist_days,64038.0,12.099,3.322,6.555,9.136,11.857,13.922,26.181
promised_days_per_100km,63708.0,20.159,74.496,0.116,3.232,5.294,10.083,3908.933


On their own, do the new features separate late from on-time orders? Late rate by fifth of each feature:

In [6]:
rows = {}
for col in ["route_slack_days", "promised_days_per_100km", "seller_late_rate", "sys_late_14d"]:
    rows[col] = train.groupby(pd.qcut(train[col], 5, duplicates="drop"), observed=True)[config.TARGET].mean().values
pd.DataFrame(rows, index=[f"fifth {i}" for i in range(1, 6)]).style.format("{:.1%}")

,route_slack_days,promised_days_per_100km,seller_late_rate,sys_late_14d
fifth 1,12.8%,12.2%,5.9%,3.5%
fifth 2,9.5%,8.8%,6.8%,4.1%
fifth 3,7.2%,8.7%,7.2%,10.8%
fifth 4,5.7%,6.2%,8.8%,12.9%
fifth 5,4.7%,3.8%,11.1%,8.6%


**Takeaway.** On their own, three of the new features separate late from on-time orders clearly. Orders with the tightest route slack were late 12.8% of the time against 4.7% for the loosest. For promised days per 100 km it was 12.2% against 3.8%, and orders from the riskiest sellers were late 11.1% of the time against 5.9%. Recent system-wide lateness has no clean pattern (3.5% rising to 12.9%, then back to 8.6%). The question is whether any of this is new information for the model.

## 3. Do they improve the models?

Same preprocessing as section 3.4 (copied from Fidan's 4.1 so the numbers line up). New numeric features are filled with the training median, logged where they are skewed, then scaled. Logistic regression and gradient boosting use Fidan's untuned settings, so any change comes from the features alone.

In [7]:
LOG_COLS = ["total_price", "seller_count", "total_freight", "item_count", "distinct_product_count",
            "total_product_volume_cm3", "approval_lag_hours", "freight_to_price_ratio", "promised_days_from_approval"]
DIST_COLS, WEIGHT_COLS = ["mean_distance_km", "max_distance_km"], ["mean_product_weight_g", "max_product_weight_g"]
OTHER_NUM = ["purchase_month", "purchase_weekday", "purchase_hour", "same_state_item_share"]
CAT_COLS = ["customer_state", "any_seller_state", "primary_product_category"]


def make_pre(extra=(), extra_log=()):
    """Section 3.4 pipeline, plus optional new features (filled, scaled; some also logged)."""
    steps = [
        ("log_scale", Pipeline([("log", FunctionTransformer(np.log1p)), ("scale", StandardScaler())]), LOG_COLS),
        ("fill_log_scale", Pipeline([("fill", SimpleImputer(strategy="median")),
                                     ("log", FunctionTransformer(np.log1p)), ("scale", StandardScaler())]),
         DIST_COLS + WEIGHT_COLS),
        ("dist_flag", MissingIndicator(features="all"), DIST_COLS),
        ("scale", StandardScaler(), OTHER_NUM),
        ("categories", Pipeline([("fill", SimpleImputer(strategy="constant", fill_value="unknown")),
                                 ("onehot", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=100))]),
         CAT_COLS),
    ]
    if extra:
        steps.append(("new", Pipeline([("fill", SimpleImputer(strategy="median")), ("scale", StandardScaler())]),
                      list(extra)))
    if extra_log:
        steps.append(("new_log", Pipeline([("fill", SimpleImputer(strategy="median")),
                                           ("log", FunctionTransformer(np.log1p)), ("scale", StandardScaler())]),
                      list(extra_log)))
    return ColumnTransformer(steps, sparse_threshold=0)


b1 = Pipeline([("pre", ColumnTransformer([
    ("days", Pipeline([("log", FunctionTransformer(np.log1p)), ("scale", StandardScaler())]),
     ["promised_days_from_approval"]),
    ("state", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=100), ["customer_state"]),
], sparse_threshold=0)), ("clf", LogisticRegression(max_iter=1000, random_state=RS))])
lr = LogisticRegression(C=1.0, max_iter=2000, random_state=RS)
hgb = HistGradientBoostingClassifier(learning_rate=0.05, max_iter=300, max_leaf_nodes=31, min_samples_leaf=50,
                                     early_stopping=False, random_state=RS)

HISTORY = ["sys_late_14d", "state_late_30d", "seller_late_rate"]
SETS = {
    "current features": dict(),
    "+ history": dict(extra=HISTORY, extra_log=["seller_past_orders"]),
    "+ route slack": dict(extra=["route_slack_days", "route_slack_ratio", "route_hist_days"]),
    "+ promised days per 100 km": dict(extra_log=["promised_days_per_100km"]),
}
models = {"B1 (promised days + state)": b1}
for set_name, kw in SETS.items():
    models[f"Logistic regression, {set_name}"] = Pipeline([("pre", make_pre(**kw)), ("clf", clone(lr))])
    models[f"HistGB, {set_name}"] = Pipeline([("pre", make_pre(**kw)), ("clf", clone(hgb))])

y = train[config.TARGET]
FOLDS = list(cv_splitter().split(train))


def cv_scores(model, folds):
    """Per-fold PR-AUC, lift and recall in the top 10%, on the given folds."""
    out = []
    for k, (tr, va) in enumerate(folds, 1):
        s = clone(model).fit(train.iloc[tr], y.iloc[tr]).predict_proba(train.iloc[va])[:, 1]
        yv = y.iloc[va]
        ap = average_precision_score(yv, s)
        out.append({"fold": k, "late_rate": yv.mean(), "pr_auc": ap, "lift": ap / yv.mean(),
                    "recall_top10": recall_at_budget(yv.values, s, 0.10)})
    return pd.DataFrame(out)


results = pd.concat([cv_scores(m, FOLDS).assign(model=name) for name, m in models.items()], ignore_index=True)
summary = (results.groupby("model", sort=False)
           .agg(pr_auc=("pr_auc", "mean"), lift=("lift", "mean"), lift_std=("lift", "std"),
                recall_top10=("recall_top10", "mean"))
           .round(3))
summary

,pr_auc,lift,lift_std,recall_top10
model,,,,
B1 (promised days + state),0.150,1.762,0.348,0.201
"Logistic regression, current features",0.158,1.808,0.214,0.215
"HistGB, current features",0.139,1.538,0.243,0.171
"Logistic regression, + history",0.153,1.736,0.292,0.198
"HistGB, + history",0.145,1.617,0.222,0.187
"Logistic regression, + route slack",0.161,1.854,0.309,0.221
"HistGB, + route slack",0.151,1.641,0.304,0.184
"Logistic regression, + promised days per 100 km",0.158,1.815,0.226,0.218
"HistGB, + promised days per 100 km",0.140,1.535,0.273,0.180


In [8]:
# Per-fold lift, to see whether any gain is consistent or comes from a single fold
results.pivot(index="model", columns="fold", values="lift").reindex(models.keys()).round(2)

fold,1,2,3,4,5
model,,,,,
B1 (promised days + state),1.43,2.24,1.45,1.98,1.71
"Logistic regression, current features",1.69,2.15,1.65,1.66,1.89
"HistGB, current features",1.17,1.80,1.67,1.43,1.62
"Logistic regression, + history",1.62,2.17,1.66,1.39,1.83
"HistGB, + history",1.33,1.92,1.64,1.49,1.71
"Logistic regression, + route slack",1.71,2.36,1.63,1.63,1.95
"HistGB, + route slack",1.14,1.90,1.62,1.67,1.87
"Logistic regression, + promised days per 100 km",1.70,2.18,1.65,1.66,1.88
"HistGB, + promised days per 100 km",1.13,1.83,1.69,1.41,1.62


**Takeaway.** None of the new features gives a real improvement.
- **History features make logistic regression slightly worse** (lift 1.81 to 1.74). Recent lateness and seller track records don't carry forward to the orders being scored.
- **Route slack is the only gain** (1.81 to 1.85), but it is smaller than the fold-to-fold spread (±0.31) and comes mostly from fold 2.
- **Promised days per 100 km changes nothing** (1.81 to 1.82). The model already has promised days and distance as separate inputs, so the ratio adds no new information.
- **Gradient boosting improves a little with each set but stays below both logistic regression and B1** every time.

**Decision:** keep the 20 features from section 3. The strong patterns in the table above are already captured by promised days, distance and customer state.

## 4. How much would a random split have flattered us?

The final model from section 5 (L1 logistic regression, C = 0.157, rare categories under 20 grouped, no `purchase_month`) scored two ways on the same training data: with our time-series folds, and with ordinary shuffled, stratified 5-fold cross-validation.

In [9]:
final_cfg = Pipeline([
    ("pre", ColumnTransformer([
        ("log_scale", Pipeline([("log", FunctionTransformer(np.log1p)), ("scale", StandardScaler())]), LOG_COLS),
        ("fill_log_scale", Pipeline([("fill", SimpleImputer(strategy="median")),
                                     ("log", FunctionTransformer(np.log1p)), ("scale", StandardScaler())]),
         DIST_COLS + WEIGHT_COLS),
        ("dist_flag", MissingIndicator(features="all"), DIST_COLS),
        ("scale", StandardScaler(), ["purchase_weekday", "purchase_hour", "same_state_item_share"]),
        ("categories", Pipeline([("fill", SimpleImputer(strategy="constant", fill_value="unknown")),
                                 ("onehot", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=20))]),
         CAT_COLS),
    ], sparse_threshold=0)),
    ("clf", LogisticRegression(solver="liblinear", penalty="l1", C=0.15728326259078687, max_iter=5000,
                               random_state=RS)),
])
random_folds = list(StratifiedKFold(5, shuffle=True, random_state=RS).split(train, y))
split_check = pd.DataFrame({
    "time-series folds (ours)": cv_scores(final_cfg, FOLDS)[["pr_auc", "recall_top10"]].mean(),
    "random stratified folds": cv_scores(final_cfg, random_folds)[["pr_auc", "recall_top10"]].mean(),
}).T.round(3)
split_check

,pr_auc,recall_top10
time-series folds (ours),0.168,0.237
random stratified folds,0.189,0.277


**Takeaway.** A random split would have made the final model look better than it is: PR-AUC 0.189 instead of 0.168, and 27.7% of late orders caught in the top 10% instead of 23.7%. Shuffled folds let the model learn from orders bought at the same time as the ones it is tested on, including the same waves of lateness. In real use it only ever sees the past, so the time-series numbers are the honest ones. This is why the project uses a time split (D6, D7).

## 5. Summary for README 4.4

- **Feature round.** After the error analysis, we tested 8 extra features in four groups: recent lateness overall and by state, seller history, route slack, and promised days per 100 km. All were built only from information available before purchase. None improved the models beyond the fold-to-fold noise, so the 20 features from section 3 were kept.
- **What this tells us.** The approval-time signal is already captured by promised days, distance and customer state. Most of what makes an order late happens after approval (carriers, strikes, volume spikes), and none of that is in the data.
- **Split check.** A random split would have overstated the final model (PR-AUC 0.189 vs 0.168, top-10% recall 27.7% vs 23.7%), which confirms the time split was the right call.
- **What did work** came later, in section 5: dropping `purchase_month`.